# Aula : Detecção de Objetos com YOLO

Nesta aula, aprenderemos sobre a arquitetura **YOLO (You Only Look Once)**, seu funcionamento interno, e como realizar o ajuste fino (fine-tuning) em um dataset real do Kaggle.

## 1. Como o YOLO Funciona?

O YOLO revolucionou a área de visão computacional ao tratar a detecção de objetos como um **problema de regressão único**, mapeando diretamente os pixels da imagem para as coordenadas da bounding box e probabilidades de classe.

### Diferencial do YOLO
Antes do YOLO, algoritmos como R-CNN propunham regiões de interesse primeiro e depois classificavam cada região de forma independente. Esse processo era lento e custoso computacionalmente.

O YOLO faz isso de uma só vez:
1. **Rede Neural Única**: Passa a imagem inteira pela rede apenas uma vez (*You Only Look Once*).
2. **Divisão em Grade**: Divide a imagem em uma grade $S \times S$.
3. **Predições da Grade**: Cada célula da grade é responsável por prever $B$ bounding boxes, o nível de confiança (*confidence score*) para cada caixa, e as probabilidades de classes condicionais.
4. **Non-Maximum Suppression (NMS)**: Remove caixas duplicadas ou com baixa confiança para obter as detecções finais.

## 2. Preparação do Ambiente e Dependências

A biblioteca moderna para trabalhar com YOLO (do YOLOv8 em diante) é a `ultralytics`. Vamos instalá-la no ambiente.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import cv2
from PIL import Image
import kagglehub
import os
import yaml
import torch
import glob

## 3. Inferência Básica com YOLO Pré-treinado

Vamos carregar um modelo YOLO pré-treinado na base COCO (por exemplo, `yolov8n.pt` que é o modelo nano, muito leve e rápido) e rodar uma inferência em uma imagem de exemplo.

In [ ]:
model = YOLO('yolov8n.pt') # Carrega o modelo pré-treinado

results = model('https://ultralytics.com/images/bus.jpg') #  inferência em uma imagem de exemplo

for r in results:
    im_array = r.plot()  # plota a imagem com as caixas delimitadoras
    im = Image.fromarray(im_array[..., ::-1])  # Converte BGR para RGB
    display(im)

## 5. Baixando um Dataset de Exemplo do Kaggle
Para fazer o fine-tuning, precisamos de um dataset. No Kaggle, existem diversos datasets formatados para YOLO (com imagens e arquivos `.txt` contendo as classes e coordenadas relativas).

Vamos baixar um dataset de exemplo no formato YOLO. Um exemplo muito comum é o dataset de detecção de capacetes de segurança ou detecção de placas de trânsito.

Para este guia, vamos baixar um dataset genérico estruturado para YOLO (Substitua a URL do dataset abaixo pelo dataset de sua escolha no Kaggle).

In [4]:
path = kagglehub.dataset_download("aneesarom/rider-with-helmet-without-helmet-number-plate") # download do dataset

100%|██████████| 74.6M/74.6M [00:00<00:00, 146MB/s]

Extracting files...


## 6. Configurando o Arquivo `data.yaml`

O YOLO precisa de um arquivo de configuração `.yaml` que indica onde estão as pastas de treino, validação, o número de classes e seus respectivos nomes.

Normalmente o dataset baixado já vem com um arquivo `.yaml` pronto, mas caso precise criar ou ajustar os caminhos absoluto das imagens, fazemos assim:

In [5]:
dataset_dir = path

train_contents = os.listdir(os.path.join(dataset_dir, 'train'))

data_config = {
    'path': dataset_dir,
    'train': 'train/images' if 'images' in train_contents else 'train',
    'val': 'val/images' if os.path.exists(os.path.join(dataset_dir, 'val/images')) else 'val',
    'nc': 4,
    'names': ['with helmet', 'without helmet', 'rider', 'number plate']
}

os.makedirs('/content/dataset', exist_ok=True)
with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("Configuração corrigida com 4 classes e salva em /content/dataset/data.yaml:", data_config)

Configuração corrigida com 4 classes e salva em /content/dataset/data.yaml: {'path': '/root/.cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate/versions/3', 'train': 'train/images', 'val': 'val/images', 'nc': 4, 'names': ['with helmet', 'without helmet', 'rider', 'number plate']}


## 7. Fine-Tuning do Modelo YOLO

Com o dataset pronto e o arquivo `.yaml` configurado, realizar o fine-tuning com a biblioteca Ultralytics é extremamente simples. Basta carregar o peso inicial e chamar a função `train`.

In [ ]:
model = YOLO('yolov8n.pt') # Carrega o modelo

device_to_use = 0 if torch.cuda.is_available() else 'cpu'

# Inicia o treinamento
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=10,                      # Número de épocas
    imgsz=640,                      # Tamanho da imagem
    batch=16,                       # Batch size
    device=device_to_use            # Usa GPU se disponível, senão CPU
)

## 8. Avaliação e Teste do Modelo Treinado

Depois que o treinamento for concluído, os melhores pesos salvos estarão salvos em `runs/detect/train/weights/best.pt`.

Podemos usar esses pesos para validar nosso modelo e fazer predições em novas imagens.


### Equações das métricas mAP50 e mAP50-95

A **Average Precision (AP)** corresponde à área sob a curva Precisão–Revocação (*Precision–Recall*), sendo definida por:

$$
AP = \int_0^1 P(R)\,dR
$$

em que \(P\) representa a precisão e \(R\) representa a revocação (*Recall*).

**1. mAP50**

A métrica mAP50 calcula a média da AP entre as \(C\) classes, considerando um limiar de IoU igual a 0,50:

$$
\boxed{
mAP_{50} = \frac{1}{C}\sum_{c=1}^{C} AP_c(0.50)
}
$$

Quanto maior o mAP50, melhor o desempenho geral do modelo nessa condição de avaliação.

**2. mAP50-95**

A métrica mAP50-95 calcula a média da AP entre as \(C\) classes e os 10 limiares de IoU, de 0,50 a 0,95, com incrementos de 0,05:

$$
\boxed{
mAP_{50:95} =
\frac{1}{10C}
\sum_{j=0}^{9}
\sum_{c=1}^{C}
AP_c(0.50+0.05j)
}
$$

em que:

- $C$: número de classes avaliadas.
- $AP_c(t)$: Average Precision da classe \(c\) para o limiar de IoU \(t\).
- $IoU$: razão entre a área de interseção e a área de união das caixas delimitadoras prevista e real.

O IoU é calculado por:

$$
IoU = \frac{A_{\text{interseção}}}
{A_{\text{união}}}
$$

Por exigir correspondências cada vez mais precisas entre as caixas previstas e as reais, o mAP50-95 oferece uma avaliação mais rigorosa da qualidade das detecções.

**Interpretação:** O mAP50 avalia as detecções considerando IoU ≥ 0,50, enquanto o mAP50-95 faz a média dos resultados em dez limiares. Assim, o mAP50-95 avalia de forma mais rigorosa a precisão espacial das caixas delimitadoras.
Ambas as métricas variam de 0 a 1 (ou de 0% a 100%), e valores mais altos indicam melhor desempenho.

In [ ]:
custom_model = YOLO('runs/detect/train-3/weights/best.pt') # Carregar o melhor modelo treinado

# Avaliar a performance no conjunto de validação
metrics = custom_model.val()

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")

Ultralytics 8.4.182 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2152.6±1036.5 MB/s, size: 797.7 KB)
val: Scanning /root/.cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate/versions/3/val/labels.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 3.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.1s/it 8.2s
                   all         20         73      0.878      0.852      0.894        0.7
           with helmet         12         13      0.968      0.769      0.863      0.633
        without helmet         10         15      0.786        0.8      0.821      0.636
                 rider         20         23      0.759      0.913      0.927       0.79
          number plate         20         22          1     

In [ ]:

# Buscar imagens de teste/validação do dataset
val_images_path = os.path.join(dataset_dir, 'val/images')
image_files = glob.glob(os.path.join(val_images_path, '*.jpg')) + glob.glob(os.path.join(val_images_path, '*.png'))

sample_images = image_files[:3] # Selecionar até 3 imagens para demonstração
results = custom_model(sample_images) # Executa a predição

for i, r in enumerate(results):
    im_array = r.plot()  # Plota as caixas e labels na imagem
    im = Image.fromarray(im_array[..., ::-1])  # Converte de BGR para RGB
    display(im)